# argentina.montos — Pruebas interactivas

Parseo de strings monetarios → número. Inverso de `arg.formato.pesos`. Lógica pura, sin dataset.

In [1]:
import argentina as arg
print(f"argentina v{arg.__version__}")

argentina v0.3.0


## 1. Parseo básico

In [2]:
casos = [
    "$ 1.500.000,50",
    "$1.500.000",
    "1500000.50",
    "ARS 1.500.000",
    "1,5M",
    "1.5 millones",
    "500 mil",
    "-1.500,50",
    "no es un monto",
    None,
]
for v in casos:
    print(f"{v!r:25s} → {arg.montos.parsear(v)}")

'$ 1.500.000,50'          → 1500000.5
'$1.500.000'              → 1500000.0
'1500000.50'              → 1500000.5
'ARS 1.500.000'           → 1500000.0
'1,5M'                    → 1500000.0
'1.5 millones'            → 1500000.0
'500 mil'                 → 500000.0
'-1.500,50'               → -1500.5
'no es un monto'          → None
None                      → None


## 2. Detección de formato

In [3]:
for v in ["1.500.000,50", "1,500,000.50", "1500000", "1.500", "1.5"]:
    print(f"{v!r:15s} → {arg.montos.formato_detectado(v)}")

'1.500.000,50'  → argentino
'1,500,000.50'  → ingles
'1500000'       → entero
'1.500'         → ambiguo
'1.5'           → ingles


## 3. Ambigüedad: `"1.500"` puede ser 1.5 o 1500

In [4]:
print("default (asume AR):  ", arg.montos.parsear("1.500"))
print("asumir='ingles':      ", arg.montos.parsear("1.500", asumir="ingles"))
print("estricto (None):      ", arg.montos.parsear_estricto("1.500"))

default (asume AR):   1500.0
asumir='ingles':       1.5
estricto (None):       None


## 4. Detección de moneda (solo cuando viene marcada inequívocamente)

In [5]:
for v in ["u$s 1.500", "USD 100", "100 dolares", "ARS 100", "100 pesos", "$ 1.500"]:
    print(f"{v!r:15s} → {arg.montos.moneda_detectada(v)}")

'u$s 1.500'     → USD
'USD 100'       → USD
'100 dolares'   → USD
'ARS 100'       → ARS
'100 pesos'     → ARS
'$ 1.500'       → None


## 5. Parseo completo: valor + moneda + formato

In [6]:
arg.montos.parsear_completo("u$s 1.500,50")

Monto(valor=1500.5, moneda='USD', formato_detectado='argentino')

## 6. Precisión decimal con `Decimal`

In [7]:
arg.montos.parsear_decimal("0,1") + arg.montos.parsear_decimal("0,2")
# Decimal('0.3'), no 0.30000000000000004

Decimal('0.3')

## 7. Round-trip con `formato.pesos`

In [8]:
for n in [1500000, 0, -1000, 1234.5]:
    s = arg.formato.pesos(n, decimales=2)
    p = arg.montos.parsear(s)
    print(f"{n!s:>10} → {s!s:>20} → {p}")

   1500000 →       $ 1.500.000,00 → 1500000.0
         0 →               $ 0,00 → 0.0
     -1000 →          -$ 1.000,00 → -1000.0
    1234.5 →           $ 1.234,50 → 1234.5


## 8. Reexport desde `formato` (descubribilidad bidireccional)

In [9]:
arg.formato.parsear_pesos("$ 1.500,50")

1500.5